# Orders.csv

In [13]:
import duckdb

con = duckdb.connect()

path = "../../source_database/seed_data/orders.csv"

### 1. Schema audit

In [15]:
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_ref             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_channel         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_status          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_date            │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ completed_at          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ total_amount          │ DOUBLE      │ 

### 2. Completeness audit

In [ ]:
# ── Completeness Audit: how many NULLs does each column have? ──

# Step 1: Get the list of column names automatically from the file.
# This way the same audit code works on ANY csv
columns_df = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
column_names = columns_df['column_name'].tolist()

# Step 2: Get the total row count once, so we can turn null counts into %.
total_rows = con.sql(f"SELECT COUNT(*) AS cnt FROM read_csv_auto('{path}')").df()['cnt'][0]
print(f"Total rows in file: {total_rows}")

# Step 3: For each column, ask DuckDB:
#   COUNT(*)        -> counts every row (NULL or not)
#   COUNT(col_name) -> counts only NON-null rows for that column
#   so COUNT(*) - COUNT(col_name) = how many rows are NULL in that column
null_report = []
for col in column_names:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) - COUNT({col}) AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT({col})) / COUNT(*), 2) AS null_percentage
        FROM read_csv_auto('{path}')
    """).df()
    null_report.append(result)

# Step 4: Stack all the per-column results into one table and show it.
import pandas as pd
null_report_df = pd.concat(null_report, ignore_index=True)
null_report_df

Total rows in file: 1228


,column_name,null_count,null_percentage
0,order_id,0,0.00
1,order_ref,0,0.00
2,customer_id,17,1.38
3,store_id,0,0.00
4,order_channel,0,0.00
5,order_status,0,0.00
6,order_date,0,0.00
7,completed_at,0,0.00
8,total_amount,0,0.00
9,discount_amount,0,0.00


### 3. Uniqueness / primary key audit

In [ ]:
# ── Uniqueness Audit: does order_id actually behave like a primary key? ──

# Step 1: Group all rows by order_id, and count how many rows fall into each group.
#   - If order_id is truly unique, every group will have exactly 1 row.
#   - HAVING COUNT(*) > 1 keeps only the groups with MORE than one row —
#     i.e. the duplicate order_ids.
duplicates_df = con.sql(f"""
    SELECT
        order_id,
        COUNT(*) AS row_count
    FROM read_csv_auto('{path}')
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
""").df()

print(f"Number of duplicate order_id values found: {len(duplicates_df)}")
duplicates_df

Number of duplicate order_id values found: 0


,order_id,row_count


In [4]:
# ── Uniqueness Audit: does order_ref actually unique? ──

# Step 1: Group all rows by order_ref, and count how many rows fall into each group.
#   - If order_id is truly unique, every group will have exactly 1 row.
#   - HAVING COUNT(*) > 1 keeps only the groups with MORE than one row —
#     i.e. the duplicate order_refs.
duplicates_df = con.sql(f"""
    SELECT
        order_ref,
        COUNT(*) AS row_count
    FROM read_csv_auto('{path}')
    GROUP BY order_ref
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
""").df()

print(f"Number of duplicate order_ref values found: {len(duplicates_df)}")
duplicates_df

Number of duplicate order_ref values found: 26


,order_ref,row_count
0,NP-2025-0000183,3
1,NP-2025-0000172,3
2,NP-2025-0000113,2
3,NP-2025-0000081,2
4,NP-2025-0000111,2
5,NP-2025-0000295,2
6,NP-2025-0000327,2
7,NP-2025-0000207,2
8,NP-2025-0000089,2
9,NP-2025-0000090,2


### 4. Referential integrity audit

In [ ]:
# ── Referential Integrity Audit: does every order's customer_id exist in customers.csv? ──

customers_path = "../../source_database/seed_data/customers.csv"

# Step 1: LEFT JOIN orders to customers on customer_id.
#   - LEFT JOIN keeps every row from orders, matched or not.
#   - Where there's no matching customer, c.customer_id comes back NULL.
# Step 2: Filter to rows where:
#   - o.customer_id IS NOT NULL  -> exclude the 17 we already know are NULL (audit #2 territory)
#   - c.customer_id IS NULL      -> no matching customer was found = orphaned foreign key
orphaned_df = con.sql(f"""
    SELECT
        o.customer_id,
        COUNT(*) AS order_count
    FROM read_csv_auto('{path}') o
    LEFT JOIN read_csv_auto('{customers_path}') c
        ON o.customer_id = c.customer_id
    WHERE o.customer_id IS NOT NULL
      AND c.customer_id IS NULL
    GROUP BY o.customer_id
    ORDER BY order_count DESC
""").df()

print(f"Distinct customer_ids in orders with NO matching customer record: {len(orphaned_df)}")
print(f"Total orphaned order rows: {orphaned_df['order_count'].sum() if len(orphaned_df) else 0}")
orphaned_df

Distinct customer_ids in orders with NO matching customer record: 0
Total orphaned order rows: 0


,customer_id,order_count


In [ ]:
# ── Referential Integrity Audit: does every order's store_id exist in stores.csv? ──

stores_path = "../../source_database/seed_data/stores.csv"

# Step 1: LEFT JOIN orders to store on store_id.
#   - LEFT JOIN keeps every row from orders, matched or not.
#   - Where there's no matching customer, s.store_id comes back NULL.
# Step 2: Filter to rows where:
#   - o.store_id IS NOT NULL  
#   - s.store_id IS NULL      -> no matching customer was found = orphaned foreign key
orphaned_df = con.sql(f"""
    SELECT
        o.store_id,
        COUNT(*) AS order_count
    FROM read_csv_auto('{path}') o
    LEFT JOIN read_csv_auto('{stores_path}') s
        ON o.store_id = s.store_id
    WHERE o.store_id IS NOT NULL
      AND s.store_id IS NULL
    GROUP BY o.store_id
    ORDER BY order_count DESC
""").df()

print(f"Distinct store_ids in orders with NO matching store record: {len(orphaned_df)}")
print(f"Total orphaned order rows: {orphaned_df['order_count'].sum() if len(orphaned_df) else 0}")
orphaned_df

Distinct store_ids in orders with NO matching store record: 0
Total orphaned order rows: 0


,store_id,order_count


### 5. Validity / business-rule audit

In [38]:
# 1) Negative money values — a sale should never have a negative amount
negative_amounts_df = con.sql(f"""
    SELECT order_id, total_amount, discount_amount, vat_amount
    FROM read_csv_auto('{path}')
    WHERE total_amount < 0 OR discount_amount < 0 OR vat_amount < 0
""").df()
print(f"Orders with a negative amount field: {len(negative_amounts_df)}")

Orders with a negative amount field: 0


In [43]:
# 2) Discount bigger than the total — can't discount more than 100% of a sale
discount_too_big_df = con.sql(f"""
    SELECT order_id, total_amount, discount_amount
    FROM read_csv_auto('{path}')
    WHERE discount_amount > total_amount
""").df()

discount_too_big_df

,order_id,total_amount,discount_amount


In [40]:
# 3) completed_at earlier than order_date — an order can't finish before it started
backwards_time_df = con.sql(f"""
    SELECT order_id, order_date, completed_at
    FROM read_csv_auto('{path}')
    WHERE completed_at < order_date
""").df()
print(f"Orders where completed_at is earlier than order_date: {len(backwards_time_df)}")

Orders where completed_at is earlier than order_date: 0


In [ ]:
# 4) order_date in the future — shouldn't exist in real historical data
future_orders_df = con.sql(f"""
    SELECT order_id, order_date
    FROM read_csv_auto('{path}')
    WHERE order_date > CURRENT_DATE
""").df()
print(f"Orders with a future order_date: {len(future_orders_df)}")

Orders with a future order_date: 0


In [ ]:
# 5) What values does order_status actually contain?
# (not a pass/fail check — just surfaces the real values so you can
#  compare them against what the data dictionary/docs claim exist)
status_values_df = con.sql(f"""
    SELECT order_status, COUNT(*) AS order_count
    FROM read_csv_auto('{path}')
    GROUP BY order_status
    ORDER BY order_count DESC
""").df()
print("\nDistinct order_status values found:")
status_values_df
print(f"Orders where discount_amount exceeds total_amount: {len(discount_too_big_df)}")


Distinct order_status values found:
Orders where discount_amount exceeds total_amount: 0


### 6. Distribution / outlier audit

In [ ]:
# ── Distribution / Outlier Audit: which values are statistically unusual? ──

# Step 1: SUMMARIZE gives min/max/avg/stddev/quartiles for every column at once.
# Focus on the numeric ones (total_amount, discount_amount, vat_amount, loyalty_points_earned).
summary_df = con.sql(f"SUMMARIZE SELECT total_amount, discount_amount, vat_amount, loyalty_points_earned FROM read_csv_auto('{path}')").df()
print("Full column summary:")
summary_df[['column_name', 'min', 'max', 'avg', 'std']]

Full column summary:


,column_name,min,max,avg,std
0,total_amount,4.58,180.59,95.91894951140047,51.273905279666955
1,discount_amount,0.0,20.93,5.608990228013028,4.654609562312778
2,vat_amount,0.31,12.0,6.397931596091213,3.419650313870665
3,loyalty_points_earned,5,215,114.61074918566776,61.5129271808716


In [ ]:
# Step 2: IQR outlier check on total_amount specifically (the most business-critical number).
#
# How IQR works:
#   Q1 = the value below which 25% of orders fall (the "low" boundary of normal)
#   Q3 = the value below which 75% of orders fall (the "high" boundary of normal)
#   IQR = Q3 - Q1  -> the width of the "normal" middle chunk of data
#   Anything below (Q1 - 1.5*IQR) or above (Q3 + 1.5*IQR) is flagged as an outlier.
#   This 1.5x multiplier is a widely-used statistics convention, not something we invented.

outliers_df = con.sql(f"""
    WITH stats AS (
        SELECT
            QUANTILE_CONT(total_amount, 0.25) AS q1,
            QUANTILE_CONT(total_amount, 0.75) AS q3
        FROM read_csv_auto('{path}')
    )
    SELECT
        o.order_id,
        o.total_amount,
        stats.q1,
        stats.q3,
        (stats.q3 - stats.q1) AS iqr
    FROM read_csv_auto('{path}') o, stats
    WHERE o.total_amount < (stats.q1 - 1.5 * (stats.q3 - stats.q1))
       OR o.total_amount > (stats.q3 + 1.5 * (stats.q3 - stats.q1))
    ORDER BY o.total_amount DESC
""").df()

print(f"Orders flagged as statistical outliers on total_amount: {len(outliers_df)}")
outliers_df

Orders flagged as statistical outliers on total_amount: 0


,order_id,total_amount,q1,q3,iqr


### 7. Freshness / volume audit

In [ ]:
# ── Freshness / Volume Audit: is the file complete and current? ──

# Step 1: Row count + the actual date range the file covers.
freshness_df = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(order_date) AS earliest_order,
        MAX(order_date) AS latest_order,
        DATEDIFF('day', MIN(order_date), MAX(order_date)) AS days_covered
    FROM read_csv_auto('{path}')
""").df()
freshness_df

,total_rows,earliest_order,latest_order,days_covered
0,1228,2025-01-01 15:07:00,2025-12-13 22:33:00,346


In [ ]:
# Step 2: Orders broken down by month — a quick way to spot suspicious
# gaps (a month with 0 or very few orders when every other month has ~100+).
monthly_counts_df = con.sql(f"""
    SELECT
        DATE_TRUNC('month', order_date) AS month,
        COUNT(*) AS order_count
    FROM read_csv_auto('{path}')
    GROUP BY month
    ORDER BY month
""").df()
monthly_counts_df

,month,order_count
0,2025-01-01,112
1,2025-02-01,110
2,2025-03-01,123
3,2025-04-01,111
4,2025-05-01,111
5,2025-06-01,108
6,2025-07-01,109
7,2025-08-01,113
8,2025-09-01,66
9,2025-10-01,95


### 8. Format consistency audit

In [ ]:
# ── Format Consistency Audit: same value, inconsistent spelling/casing? ──

text_columns = ['order_channel', 'order_status', 'payment_method']

consistency_report = []
for col in text_columns:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(DISTINCT {col}) AS raw_distinct_count,
            COUNT(DISTINCT LOWER(TRIM({col}))) AS normalized_distinct_count
        FROM read_csv_auto('{path}')
    """).df()
    consistency_report.append(result)

import pandas as pd
consistency_df = pd.concat(consistency_report, ignore_index=True)

# If raw > normalized, some values are duplicates of each other under
# different casing/spacing — this column shows exactly how many.
consistency_df['extra_variants'] = (
    consistency_df['raw_distinct_count'] - consistency_df['normalized_distinct_count']
)
consistency_df

,column_name,raw_distinct_count,normalized_distinct_count,extra_variants
0,order_channel,3,3,0
1,order_status,3,3,0
2,payment_method,4,4,0


In [ ]:
# For any column above with extra_variants > 0, list its actual raw values
# so you can see the exact inconsistency (e.g. "Online" vs "online" vs " Online").
for col in text_columns:
    values_df = con.sql(f"""
        SELECT DISTINCT {col}
        FROM read_csv_auto('{path}')
        ORDER BY {col}
    """).df()
    print(f"\n{col} — {len(values_df)} distinct raw value(s):")
    print(values_df[col].tolist())


order_channel — 3 distinct raw value(s):
['click_collect', 'in_store', 'online']

order_status — 3 distinct raw value(s):
['cancelled', 'completed', 'returned']

payment_method — 4 distinct raw value(s):
['card', 'cash', 'contactless', 'online']


# 9. Distribution anomaly check

In [14]:
# 9. Distribution anomaly check
for col in ['order_channel', 'payment_method', 'is_returned']:
    print(f"\n-- {col} --")
    print(con.sql(f"""
        SELECT {col}, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct
        FROM read_csv_auto('{path}') GROUP BY {col} ORDER BY n DESC
    """).df())




-- order_channel --
   order_channel    n   pct
0       in_store  785  63.9
1  click_collect  240  19.5
2         online  203  16.5

-- payment_method --
  payment_method    n   pct
0           card  615  50.1
1    contactless  220  17.9
2         online  199  16.2
3           cash  194  15.8

-- is_returned --
   is_returned     n   pct
0        False  1192  97.1
1         True    36   2.9


# 10. Timezone drift: hour-of-day distribution

In [35]:
# --- 10. Timezone drift: hour-of-day distribution ---
# order_date
print("order_date")
print(con.sql(f"""
    SELECT EXTRACT(hour FROM order_date) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())

# completed_at
print("\ncompleted_at")
print(con.sql(f"""
    SELECT EXTRACT(hour FROM completed_at) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())

# created_at
print("\ncreated_at")
print(con.sql(f"""
    SELECT EXTRACT(hour FROM created_at) AS hour, COUNT(*) AS n
    FROM read_csv_auto('{path}') GROUP BY hour ORDER BY hour
""").df())


#  Logical time order: nothing should be "completed" before it was "ordered" ---
print("\n Logical time order: nothing should be 'completed' before it was 'ordered'")
print(con.sql(f"""
    SELECT COUNT(*) AS backwards_count
    FROM read_csv_auto('{path}') WHERE completed_at < order_date
""").df())


#  Logical time order: nothing should be "completed" before it was "ordered" ---
print("\n Logical time order: nothing should be 'completed' before it was 'created'")
print(con.sql(f"""
    SELECT COUNT(*) AS backwards_count
    FROM read_csv_auto('{path}') WHERE completed_at < created_at
""").df())

order_date
    hour   n
0      6  81
1      7  85
2      8  72
3      9  62
4     10  80
5     11  71
6     12  83
7     13  69
8     14  66
9     15  60
10    16  82
11    17  57
12    18  63
13    19  73
14    20  65
15    21  80
16    22  79

completed_at
    hour   n
0      6  56
1      7  85
2      8  71
3      9  62
4     10  85
5     11  64
6     12  78
7     13  71
8     14  72
9     15  64
10    16  77
11    17  61
12    18  62
13    19  73
14    20  63
15    21  78
16    22  80
17    23  26

created_at
    hour   n
0      6  81
1      7  85
2      8  72
3      9  62
4     10  80
5     11  71
6     12  83
7     13  69
8     14  66
9     15  60
10    16  82
11    17  57
12    18  63
13    19  73
14    20  65
15    21  80
16    22  79

 Logical time order: nothing should be 'completed' before it was 'ordered'
   backwards_count
0                0

 Logical time order: nothing should be 'completed' before it was 'created'
   backwards_count
0                1


In [32]:
results = []

# ORDER_DATE
# 1. Hour-of-day spread (timezone drift check)
hours = con.sql(f"SELECT EXTRACT(hour FROM order_date) AS hour, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY hour").df()
results.append({
    'check': 'order_date hour-of-day coverage',
    'result': f"{hours['hour'].nunique()} distinct hours used (range {int(hours['hour'].min())}-{int(hours['hour'].max())})",
    'status': 'OK'
})

# 2. Is updated_at a real timestamp or a constant placeholder?
stats = con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT order_date) AS distinct_dates
    FROM read_csv_auto('{path}')
""").df().iloc[0]

results.append({
    'check': 'order_date distinct values',
    'result': f"{stats['distinct_dates']} distinct value(s) across {stats['total_rows']} rows",
    'status': 'OK' if stats['distinct_dates'] > 1 else '⚠️ FLAG — placeholder, not real'
})

# 3. Gaps in order_id sequence
gaps = con.sql(f"""
    WITH bounds AS (SELECT MIN(order_id) mn, MAX(order_id) mx FROM read_csv_auto('{path}')),
    spine AS (SELECT UNNEST(generate_series(mn, mx)) AS id FROM bounds)
    SELECT COUNT(*) AS n FROM spine s LEFT JOIN read_csv_auto('{path}') c ON s.id = c.order_id WHERE c.order_id IS NULL
""").fetchone()[0]
results.append({
    'check': 'order_id sequence gaps',
    'result': f"{gaps} missing ID(s)",
    'status': 'OK' if gaps == 0 else '⚠️ FLAG'
})

# COMPLETED_AT
# 1. Hour-of-day spread (timezone drift check)
hours = con.sql(f"SELECT EXTRACT(hour FROM completed_at) AS hour, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY hour").df()
results.append({
    'check': 'completed_at hour-of-day coverage',
    'result': f"{hours['hour'].nunique()} distinct hours used (range {int(hours['hour'].min())}-{int(hours['hour'].max())})",
    'status': 'OK'
})

# 2. Is completed_at a real timestamp or a constant placeholder?
stats = con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT completed_at) AS distinct_dates
    FROM read_csv_auto('{path}')
""").df().iloc[0]

results.append({
    'check': 'completed_at distinct values',
    'result': f"{stats['distinct_dates']} distinct value(s) across {stats['total_rows']} rows",
    'status': 'OK' if stats['distinct_dates'] > 1 else '⚠️ FLAG — placeholder, not real'
})

# CREATED_AT
# 1. Hour-of-day spread (timezone drift check)
hours = con.sql(f"SELECT EXTRACT(hour FROM created_at) AS hour, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY hour").df()
results.append({
    'check': 'created_at hour-of-day coverage',
    'result': f"{hours['hour'].nunique()} distinct hours used (range {int(hours['hour'].min())}-{int(hours['hour'].max())})",
    'status': 'OK'
})

# 2. Is updated_at a real timestamp or a constant placeholder?
stats = con.sql(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT created_at) AS distinct_dates
    FROM read_csv_auto('{path}')
""").df().iloc[0]

results.append({
    'check': 'created_at distinct values',
    'result': f"{stats['distinct_dates']} distinct value(s) across {stats['total_rows']} rows",
    'status': 'OK' if stats['distinct_dates'] > 1 else '⚠️ FLAG — placeholder, not real'
})

summary = pd.DataFrame(results)
summary

,check,result,status
0,order_date hour-of-day coverage,17 distinct hours used (range 6-22),OK
1,order_date distinct values,1198 distinct value(s) across 1228 rows,OK
2,order_id sequence gaps,0 missing ID(s),OK
3,completed_at hour-of-day coverage,18 distinct hours used (range 6-23),OK
4,completed_at distinct values,1199 distinct value(s) across 1228 rows,OK
5,created_at hour-of-day coverage,17 distinct hours used (range 6-22),OK
6,created_at distinct values,1226 distinct value(s) across 1228 rows,OK


# 11. Format consistency: Flag columns that contain mixed data types

In [36]:
# 11. Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,order_id,INTEGER,1228,False
1,order_ref,TEXT,1228,False
2,customer_id,INTEGER,1211,False
3,customer_id,NULL/EMPTY,17,False
4,store_id,INTEGER,1228,False
5,order_channel,TEXT,1228,False
6,order_status,TEXT,1228,False
7,order_date,DATE,1228,False
8,completed_at,DATE,1228,False
9,total_amount,INTEGER,1228,False


# 12. Sentinel/Default value usage:

In [37]:
# 12. Sentinel/Default value usage:

# Part 1: scan for known/common sentinel patterns
SENTINEL_VALUES = [
    '-1', '0', '999', '9999', '99999', '-999',
    'NA', 'N/A', 'NULL', 'NONE', 'UNKNOWN', 'UNK', 'TBD', '-', '?',
    '1900-01-01', '1970-01-01', '9999-12-31', '0001-01-01'
]

def sentinel_scan(con, path, sentinel_values):
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()
    values_sql = ", ".join(f"'{v.upper()}'" for v in sentinel_values)

    all_rows = []
    for col in columns:
        df = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                "{col}" AS value,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            WHERE upper(TRIM("{col}")) IN ({values_sql})
            GROUP BY "{col}"
            ORDER BY row_count DESC
        """).df()
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
sentinel_report


,column_name,value,row_count
0,order_id,999,1


# Orders.xlsx

In [21]:
# SETUP
import duckdb
import pandas as pd

path = "../../source_database/seed_data/orders.xlsx"  # adjust if needed
orders_xlsx = pd.read_excel(path)

# side-step DuckDB's register() dtype issue — export to CSV, then read that
orders_xlsx.to_csv("orders_xlsx_export.csv", index=False)

con = duckdb.connect()
xlsx_path = "orders_xlsx_export.csv"

In [22]:
# 1. SCHEMA AUDIT
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{xlsx_path}')").show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_ref         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_channel     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_status      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_date        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ completed_at      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ total_amount      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ payment_me

In [23]:
# 2. COMPLETENESS AUDIT
total_rows = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{xlsx_path}')").fetchone()[0]
print(f"Total rows: {total_rows}")

cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{xlsx_path}')").df()['column_name'].tolist()
for col in cols:
    null_count = con.sql(f"SELECT COUNT(*) - COUNT({col}) FROM read_csv_auto('{xlsx_path}')").fetchone()[0]
    pct = round(null_count / total_rows * 100, 2)
    print(f"{col}: {null_count} nulls ({pct}%)")

Total rows: 1256
order_id: 0 nulls (0.0%)
order_ref: 0 nulls (0.0%)
customer_id: 59 nulls (4.7%)
store_id: 0 nulls (0.0%)
order_channel: 0 nulls (0.0%)
order_status: 0 nulls (0.0%)
order_date: 0 nulls (0.0%)
completed_at: 0 nulls (0.0%)
total_amount: 0 nulls (0.0%)
payment_method: 0 nulls (0.0%)
payment_reference: 0 nulls (0.0%)
is_returned: 0 nulls (0.0%)
week_number: 0 nulls (0.0%)
year: 0 nulls (0.0%)


In [24]:
# 3. UNIQUENESS AUDIT — order_id and order_ref
dupe_ids = con.sql(f"""
    SELECT order_id, COUNT(*) AS row_count
    FROM read_csv_auto('{xlsx_path}')
    GROUP BY order_id
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicate order_id values: {len(dupe_ids)}")

dupe_refs = con.sql(f"""
    SELECT order_ref, COUNT(*) AS row_count
    FROM read_csv_auto('{xlsx_path}')
    GROUP BY order_ref
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
""").df()
print(f"Duplicate order_ref values: {len(dupe_refs)}")
dupe_refs

Duplicate order_id values: 0
Duplicate order_ref values: 28


,order_ref,row_count
0,NP-2025-0000099,2
1,NP-2025-0000332,2
2,NP-2025-0001187,2
3,NP-2025-0000298,2
4,NP-2025-0000985,2
5,NP-2025-0000024,2
6,NP-2025-0000991,2
7,NP-2025-0001172,2
8,NP-2025-0000374,2
9,NP-2025-0000488,2


In [33]:
orphan_stores = con.sql(f"""
    SELECT COUNT(*)
    FROM read_csv_auto('{xlsx_path}') o
    LEFT JOIN read_csv_auto('../../source_database/seed_data/stores.csv') s ON o.store_id = s.store_id
    WHERE s.store_id IS NULL
""").fetchone()[0]
print(f"Orphaned store_id rows: {orphan_stores}")

Orphaned store_id rows: 0


In [34]:
# 5. VALIDITY / BUSINESS RULE AUDIT
con.sql(f"""
    SELECT
        SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_amounts,
        SUM(CASE WHEN completed_at < order_date THEN 1 ELSE 0 END) AS completed_before_ordered,
        SUM(CASE WHEN order_date > CURRENT_DATE THEN 1 ELSE 0 END) AS future_order_dates
    FROM read_csv_auto('{xlsx_path}')
""").show()

con.sql(f"SELECT DISTINCT order_status FROM read_csv_auto('{xlsx_path}')").show()
con.sql(f"SELECT DISTINCT order_channel FROM read_csv_auto('{xlsx_path}')").show()
con.sql(f"SELECT DISTINCT payment_method FROM read_csv_auto('{xlsx_path}')").show()

┌──────────────────┬──────────────────────────┬────────────────────┐
│ negative_amounts │ completed_before_ordered │ future_order_dates │
│      int128      │          int128          │       int128       │
├──────────────────┼──────────────────────────┼────────────────────┤
│                0 │                        0 │                  0 │
└──────────────────┴──────────────────────────┴────────────────────┘

┌──────────────┐
│ order_status │
│   varchar    │
├──────────────┤
│ cancelled    │
│ refunded     │
│ completed    │
└──────────────┘

┌───────────────┐
│ order_channel │
│    varchar    │
├───────────────┤
│ in_store      │
│ online        │
│ click_collect │
└───────────────┘

┌────────────────┐
│ payment_method │
│    varchar     │
├────────────────┤
│ card           │
│ cash           │
│ paypal         │
└────────────────┘



In [27]:
# 6. DISTRIBUTION / OUTLIER AUDIT — IQR on total_amount
iqr = con.sql(f"""
    SELECT QUANTILE_CONT(total_amount, 0.25) AS q1, QUANTILE_CONT(total_amount, 0.75) AS q3
    FROM read_csv_auto('{xlsx_path}')
""").df()
q1, q3 = iqr['q1'][0], iqr['q3'][0]
iqr_range = q3 - q1
lower_bound = q1 - 1.5 * iqr_range
upper_bound = q3 + 1.5 * iqr_range

outliers = con.sql(f"""
    SELECT * FROM read_csv_auto('{xlsx_path}')
    WHERE total_amount < {lower_bound} OR total_amount > {upper_bound}
""").df()
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]  |  Outliers found: {len(outliers)}")
outliers

Bounds: [-135.71, 411.22]  |  Outliers found: 0


,order_id,order_ref,customer_id,store_id,order_channel,order_status,order_date,completed_at,total_amount,payment_method,payment_reference,is_returned,week_number,year


In [28]:
# 7. FRESHNESS / VOLUME AUDIT
con.sql(f"""
    SELECT MIN(order_date) AS earliest_order, MAX(order_date) AS latest_order,
           DATEDIFF('day', MIN(order_date), MAX(order_date)) AS days_covered, COUNT(*) AS total_rows
    FROM read_csv_auto('{xlsx_path}')
""").show()

con.sql(f"""
    SELECT DATE_TRUNC('month', order_date) AS month, COUNT(*) AS order_count
    FROM read_csv_auto('{xlsx_path}')
    GROUP BY month ORDER BY month
""").show()

┌─────────────────────┬─────────────────────┬──────────────┬────────────┐
│   earliest_order    │    latest_order     │ days_covered │ total_rows │
│      timestamp      │      timestamp      │    int64     │   int64    │
├─────────────────────┼─────────────────────┼──────────────┼────────────┤
│ 2025-01-01 11:30:00 │ 2025-12-31 20:04:00 │          364 │       1256 │
└─────────────────────┴─────────────────────┴──────────────┴────────────┘

┌────────────┬─────────────┐
│   month    │ order_count │
│    date    │    int64    │
├────────────┼─────────────┤
│ 2025-01-01 │         106 │
│ 2025-02-01 │          93 │
│ 2025-03-01 │         123 │
│ 2025-04-01 │         123 │
│ 2025-05-01 │          99 │
│ 2025-06-01 │          93 │
│ 2025-07-01 │          86 │
│ 2025-08-01 │         146 │
│ 2025-09-01 │         109 │
│ 2025-10-01 │          97 │
│ 2025-11-01 │          92 │
│ 2025-12-01 │          89 │
├────────────┴─────────────┤
│ 12 rows        2 columns │
└──────────────────────────┘



In [29]:
# 8. FORMAT CONSISTENCY AUDIT
for col in ["order_channel", "order_status", "payment_method"]:
    print(f"-- {col} --")
    con.sql(f"""
        SELECT DISTINCT {col}, LOWER(TRIM({col})) AS normalized
        FROM read_csv_auto('{xlsx_path}')
    """).show()

-- order_channel --
┌───────────────┬───────────────┐
│ order_channel │  normalized   │
│    varchar    │    varchar    │
├───────────────┼───────────────┤
│ online        │ online        │
│ in_store      │ in_store      │
│ click_collect │ click_collect │
└───────────────┴───────────────┘

-- order_status --
┌──────────────┬────────────┐
│ order_status │ normalized │
│   varchar    │  varchar   │
├──────────────┼────────────┤
│ cancelled    │ cancelled  │
│ refunded     │ refunded   │
│ completed    │ completed  │
└──────────────┴────────────┘

-- payment_method --
┌────────────────┬────────────┐
│ payment_method │ normalized │
│    varchar     │  varchar   │
├────────────────┼────────────┤
│ paypal         │ paypal     │
│ card           │ card       │
│ cash           │ cash       │
└────────────────┴────────────┘



In [30]:
# BONUS — compare against orders.csv
csv_count = con.sql("SELECT COUNT(*) FROM read_csv_auto('../../source_database/seed_data/orders.csv')").fetchone()[0]
xlsx_count = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{xlsx_path}')").fetchone()[0]
print(f"orders.csv rows:  {csv_count}")
print(f"orders.xlsx rows: {xlsx_count}")

orders.csv rows:  1228
orders.xlsx rows: 1256


In [31]:
import os
for f in sorted(os.listdir("../../source_database/seed_data/")):
    print(f)

.ipynb_checkpoints
customers.csv
customers.xlsx
inventory.csv
inventory.xlsx
order_items.csv
orders.csv
orders.xlsx
products.csv
products.xlsx
stores.csv
suppliers.csv


# Comparision between csv and xlsx

In [1]:
# ==========================================================
# ORDERS.CSV vs ORDERS.XLSX — FULL COMPARISON AUDIT
# Goal: find every way these two files disagree, with numbers
# ==========================================================
!pip install openpyxl
import pandas as pd

csv_path = "../../source_database/seed_data/orders.csv"
xlsx_path = "../../source_database/seed_data/orders.xlsx"

csv_df = pd.read_csv(csv_path)
xlsx_df = pd.read_excel(xlsx_path)   # needs openpyxl installed

print(f"orders.csv  -> {csv_df.shape[0]} rows, {csv_df.shape[1]} columns")
print(f"orders.xlsx -> {xlsx_df.shape[0]} rows, {xlsx_df.shape[1]} columns")

orders.csv  -> 1228 rows, 16 columns
orders.xlsx -> 1256 rows, 14 columns


In [2]:
# STEP 1 — SCHEMA COMPARISON
# A file with extra/missing columns can't be a straight reformat of the other.
csv_cols = set(csv_df.columns)
xlsx_cols = set(xlsx_df.columns)

only_in_csv  = csv_cols - xlsx_cols
only_in_xlsx = xlsx_cols - csv_cols
shared_cols  = csv_cols & xlsx_cols

print("Columns only in orders.csv :", sorted(only_in_csv))
print("Columns only in orders.xlsx:", sorted(only_in_xlsx))
print("Columns in both files      :", sorted(shared_cols))

Columns only in orders.csv : ['created_at', 'discount_amount', 'loyalty_points_earned', 'vat_amount']
Columns only in orders.xlsx: ['week_number', 'year']
Columns in both files      : ['completed_at', 'customer_id', 'is_returned', 'order_channel', 'order_date', 'order_id', 'order_ref', 'order_status', 'payment_method', 'payment_reference', 'store_id', 'total_amount']


In [3]:
# STEP 2 — DATA TYPE COMPARISON
# Same column name doesn't guarantee same data type — worth checking before joining.
dtype_compare = pd.DataFrame({
    "csv_dtype":  csv_df[sorted(shared_cols)].dtypes,
    "xlsx_dtype": xlsx_df[sorted(shared_cols)].dtypes
})
dtype_compare["same_dtype"] = dtype_compare["csv_dtype"] == dtype_compare["xlsx_dtype"]
dtype_compare

,csv_dtype,xlsx_dtype,same_dtype
completed_at,str,str,True
customer_id,float64,float64,True
is_returned,bool,bool,True
order_channel,str,str,True
order_date,str,str,True
order_id,int64,int64,True
order_ref,str,str,True
order_status,str,str,True
payment_method,str,str,True
payment_reference,str,str,True


In [4]:
# STEP 3 — order_id OVERLAP
# Does every order_id in one file exist in the other, or are there orders unique to each?
csv_ids  = set(csv_df["order_id"])
xlsx_ids = set(xlsx_df["order_id"])

only_in_csv_ids  = csv_ids - xlsx_ids
only_in_xlsx_ids = xlsx_ids - csv_ids
shared_ids       = csv_ids & xlsx_ids

print(f"order_id only in csv : {len(only_in_csv_ids)}")
print(f"order_id only in xlsx: {len(only_in_xlsx_ids)}")
print(f"order_id in both     : {len(shared_ids)}")

if only_in_xlsx_ids:
    print("order_ids only in xlsx (min-max):", min(only_in_xlsx_ids), "-", max(only_in_xlsx_ids))
if only_in_csv_ids:
    print("order_ids only in csv (min-max):", min(only_in_csv_ids), "-", max(only_in_csv_ids))

order_id only in csv : 0
order_id only in xlsx: 28
order_id in both     : 1228
order_ids only in xlsx (min-max): 1229 - 1256


In [5]:
# STEP 4 — DUPLICATE order_ref CHECK (run separately on each file)
# Confirms whether the duplication problem exists in one file, the other, or both.
def duplicate_ref_summary(df, label):
    dupes = df.groupby("order_ref").size()
    dupes = dupes[dupes > 1]
    excess_rows = dupes.sum() - len(dupes)
    pct = round(excess_rows / len(df) * 100, 2)
    print(f"{label}: {len(dupes)} duplicated order_ref values, {excess_rows} excess rows ({pct}% of file)")
    return dupes

csv_dupe_refs  = duplicate_ref_summary(csv_df, "orders.csv")
xlsx_dupe_refs = duplicate_ref_summary(xlsx_df, "orders.xlsx")

orders.csv: 26 duplicated order_ref values, 28 excess rows (2.28% of file)
orders.xlsx: 28 duplicated order_ref values, 28 excess rows (2.23% of file)


In [6]:
# STEP 5 — FIELD-BY-FIELD COMPARISON FOR order_ids PRESENT IN BOTH FILES
# This is the key check: for the SAME order_id, do the two files actually agree?
# xlsx has duplicate order_ids for order_ref, so keep only the first row per order_id before joining.
xlsx_dedup = xlsx_df.drop_duplicates(subset="order_id", keep="first")

compare_cols = sorted(shared_cols - {"order_id"})  # don't compare the join key against itself

merged = csv_df.merge(xlsx_dedup, on="order_id", suffixes=("_csv", "_xlsx"))
print(f"Rows compared (order_id present in both files): {len(merged)}")

mismatch_summary = {}
for col in compare_cols:
    mismatches = merged[f"{col}_csv"] != merged[f"{col}_xlsx"]
    mismatch_summary[col] = {
        "mismatches": int(mismatches.sum()),
        "pct": round(mismatches.sum() / len(merged) * 100, 2)
    }

mismatch_df = pd.DataFrame(mismatch_summary).T.sort_values("pct", ascending=False)
mismatch_df

Rows compared (order_id present in both files): 1228


,mismatches,pct
completed_at,1228.0,100.00
order_date,1228.0,100.00
payment_reference,1228.0,100.00
total_amount,1228.0,100.00
customer_id,1226.0,99.84
store_id,1183.0,96.34
order_channel,829.0,67.51
payment_method,808.0,65.80
order_status,724.0,58.96
is_returned,68.0,5.54


In [7]:
# STEP 7 — FINAL SUMMARY (copy this block straight into your documentation)
print("=== orders.csv vs orders.xlsx — comparison summary ===")
print(f"orders.csv : {len(csv_df)} rows, {len(csv_df.columns)} columns")
print(f"orders.xlsx: {len(xlsx_df)} rows, {len(xlsx_df.columns)} columns")
print(f"Columns only in csv : {sorted(only_in_csv)}")
print(f"Columns only in xlsx: {sorted(only_in_xlsx)}")
print(f"order_id shared between files: {len(shared_ids)}")
print(f"order_id only in xlsx: {len(only_in_xlsx_ids)}")
print()
print("Field mismatch rate on shared order_ids:")
print(mismatch_df)

=== orders.csv vs orders.xlsx — comparison summary ===
orders.csv : 1228 rows, 16 columns
orders.xlsx: 1256 rows, 14 columns
Columns only in csv : ['created_at', 'discount_amount', 'loyalty_points_earned', 'vat_amount']
Columns only in xlsx: ['week_number', 'year']
order_id shared between files: 1228
order_id only in xlsx: 28

Field mismatch rate on shared order_ids:
                   mismatches     pct
completed_at           1228.0  100.00
order_date             1228.0  100.00
payment_reference      1228.0  100.00
total_amount           1228.0  100.00
customer_id            1226.0   99.84
store_id               1183.0   96.34
order_channel           829.0   67.51
payment_method          808.0   65.80
order_status            724.0   58.96
is_returned              68.0    5.54
order_ref                28.0    2.28
